# 01 — Format-aware ingestion + tier + edition metadata

**Phase 1** of the notebook chain (plan §1.5, §2.6, §6 Stage 1).

Wires the deterministic ingest path:

`path -> tier resolver -> reader (epub/pdf/packed_md) -> PageRecord -> Neo4j DOCUMENT/PAGE (+ MinIO for image pages)`

Concretely:

1. **Tier resolver** — `raw/Primary/...` -> `primary`; `raw/Secondary/...` -> `secondary`.
2. **Format detector** — picks `epub_reader`, `pdf_reader`, or `packed_md` based on path shape.
3. **PageRecord** — uniform dataclass produced by every reader; v2.1 spine adds `chapter_id` / `section_id` fields.
4. **Edition metadata** — `extract_from_filename` (regex; ~70% coverage)
   with optional `llm_enrich` (deepseek-chat) for the long tail.
5. **Structure extractor** — `apps.backend.pipeline.structure.plan_structure`
   builds the CHAPTER/SECTION spine TOC-first (PDF `fitz.get_toc`, EPUB `book.toc`
   NCX walker, packed-md directory tree), with a synthetic single-CHAPTER/SECTION
   fallback so every page has parents. Phase-3.5 LLM fallback is deferred.
6. **Orchestrator** — `ingest_path(...)` upserts the v2.1 spine
   `(:TOPIC)-[:CONTAIN]->(:DOCUMENT)-[:CONSIST_OF]->(:CHAPTER)-[:INCLUDE]->(:SECTION)-[:INCLUDE]->(:PAGE)`,
   streams OCR-bound pages to MinIO `ancient-pages/`, and links
   consecutive pages with `[:NEXT]`.

**Inputs**

- `notebooks/_artifacts/00_setup_smoke_test/health.json` — must report all-healthy.
- `notebooks/_artifacts/00a_philological_seeds/seeds.json` — for cross-checking checksum continuity.

**Outputs**

- `notebooks/_artifacts/01_ingestion/ingestion.json` — per-document ingest reports
  (page counts, native/ocr split, MinIO uploads, metadata, errors).

**Fixtures used (subset of `raw/`)**

- `raw/Primary/唐摭言.epub` — Tang 笔记小说; mixes native HTML chapters
  with embedded scanned-image bundles, exercises the EPUB image expander.
- `raw/Secondary/specific_科举/田子爽_唐代制举孝悌类科目考论.pdf` — modern academic PDF;
  all-native text, fast PyMuPDF detection.
- `raw/Secondary/comprehensive_epub/阎步克_察举制度变迁史稿.packed/` — pre-chunked
  Markdown tree (Obsidian Epub Importer).


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

HERE = Path.cwd()
REPO_ROOT = HERE if (HERE / "AGENTS.md").exists() else HERE.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "_artifacts" / "01_ingestion"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT     =", REPO_ROOT)
print("ARTIFACT_DIR  =", ARTIFACT_DIR)
print("CHAT model    =", os.getenv("CHAT_LLM_MODEL"))
print("Neo4j URI     =", os.getenv("NEO4J_URI"))
print("MinIO endpoint=", os.getenv("MINIO_ENDPOINT"))


## 1. Cross-check prior artifacts

Phase 1 depends on Phase 0 (services healthy) and Phase 0a (philological
seeds checksummed). Read both artifact files and assert they exist; abort
early if either is missing.


In [2]:
import json

PRIOR_HEALTH = REPO_ROOT / "notebooks" / "_artifacts" / "00_setup_smoke_test" / "health.json"
PRIOR_SEEDS = REPO_ROOT / "notebooks" / "_artifacts" / "00a_philological_seeds" / "seeds.json"

assert PRIOR_HEALTH.exists(), f"missing {PRIOR_HEALTH}: run 00_setup_smoke_test.ipynb first"
assert PRIOR_SEEDS.exists(), f"missing {PRIOR_SEEDS}: run 00a_philological_seeds.ipynb first"

health = json.loads(PRIOR_HEALTH.read_text())
seeds  = json.loads(PRIOR_SEEDS.read_text())

for svc in ("silra", "neo4j", "minio"):
    ok = bool(health.get(svc, {}).get("ok"))
    print(f"  {svc:6s}: {'OK' if ok else 'FAIL'}")
    assert ok, f"service {svc} unhealthy in prior artifact: {health.get(svc)}"

seed_inv = seeds.get("seed_inventory") or {}
print(f"  seeds: {len(seed_inv)} files checksummed")


  silra : OK
  neo4j : OK
  minio : OK
  seeds: 7 files checksummed


## 2. Wire up clients + reaffirm Neo4j schema

Open the Silra / Neo4j / MinIO clients we'll reuse for the rest of the
notebook, ensure the `ancient-pages` bucket exists, and re-run
`init_schema()` so the Phase 1 lookup indexes (DOCUMENT.tier,
PAGE.tier, PAGE.docPageIndex, …) are guaranteed to exist on idempotent
re-runs.


In [3]:
from apps.backend.graph.neo4j_client import get_driver, ping as neo4j_ping
from apps.backend.graph.schema import init_schema
from apps.backend.storage.minio_client import get_minio_client, ensure_bucket
from apps.backend.llm.silra import ping as silra_ping

silra_status = silra_ping()
print("silra ping:", "OK" if silra_status.get("ok") else "FAIL", "-", silra_status)
assert silra_status.get("ok"), silra_status

driver = get_driver()
neo_status = neo4j_ping(driver=driver)
print("neo4j ping:", "OK" if neo_status.get("ok") else "FAIL", "-", neo_status)
assert neo_status.get("ok"), neo_status

bucket = os.getenv("MINIO_BUCKET_PAGES", "ancient-pages")
minio_client = get_minio_client()
ensure_bucket(minio_client, bucket)
print(f"minio bucket '{bucket}' ready")

init_report = init_schema(driver=driver)
print(f"schema: {len(init_report.get('constraints', []))} constraints, "
      f"{len(init_report.get('vector_indexes', []))} vector indexes, "
      f"{len(init_report.get('lookup_indexes', []))} lookup indexes")
assert not init_report.get("errors"), init_report["errors"]


silra ping: OK - {'ok': True, 'base_url': 'https://api.silra.cn/v1/', 'chat_model': 'deepseek-chat', 'embed_model': 'text-embedding-v4', 'ocr_model': 'deepseek-ocr', 'embedding_dims_expected': 1024, 'embedding_dims_observed': 1024, 'chat_sample': 'OK', 'errors': []}
neo4j ping: OK - {'ok': True, 'uri': 'bolt://localhost:7687', 'server_version': '5.18.1', 'edition': 'community', 'database': 'neo4j', 'constraint_count': 15, 'vector_index_count': 5, 'errors': []}
minio bucket 'ancient-pages' ready
schema: 16 constraints, 5 vector indexes, 17 lookup indexes


## 3. Tier resolver

Deterministic: every path under `raw/Primary/` is `primary`; every path
under `raw/Secondary/` (any depth) is `secondary`; anything else falls
back to `primary` and is flagged for manual review.


In [4]:
from apps.backend.readers import resolve_tier, detect_reader

cases = [
    ("raw/Primary/唐摭言.epub", "primary", "epub"),
    ("raw/Primary/通典.pdf", "primary", "pdf"),
    ("raw/Secondary/comprehensive_官序/朱博宇_xxx.pdf", "secondary", "pdf"),
    ("raw/Secondary/comprehensive_epub/阎步克_察举制度变迁史稿.packed", "secondary", "packed_md"),
]
for path, expected_tier, expected_backend in cases:
    full = REPO_ROOT / path
    # Materialise the dir for packed_md detection probe; falls back to suffix detect.
    actual_tier = resolve_tier(path)
    try:
        actual_backend = detect_reader(full) if full.exists() else (
            "packed_md" if path.endswith(".packed") else
            ("epub" if path.endswith(".epub") else "pdf")
        )
    except ValueError:
        actual_backend = "?"
    ok = actual_tier == expected_tier and actual_backend == expected_backend
    print(f"  {'OK' if ok else 'FAIL'}  tier={actual_tier:9s} backend={actual_backend:9s}  {path}")
    assert ok, (path, actual_tier, actual_backend)


  OK  tier=primary   backend=epub       raw/Primary/唐摭言.epub
  OK  tier=primary   backend=pdf        raw/Primary/通典.pdf
  OK  tier=secondary backend=pdf        raw/Secondary/comprehensive_官序/朱博宇_xxx.pdf
  OK  tier=secondary backend=packed_md  raw/Secondary/comprehensive_epub/阎步克_察举制度变迁史稿.packed


## 4. Filename → edition metadata (regex layer)

The regex pass handles ~70% of the corpus deterministically. We exercise
it on a curated cross-section of `raw/Primary/` and `raw/Secondary/` so
we can eyeball coverage of edition / 編校者 / publisher / publication-year
/ editorial-layer detection. Edge cases that fail here are the targets
for the optional LLM enrichment step (cell 8).


In [5]:
from apps.backend.pipeline.metadata import extract_from_filename

samples = [
    "册府元龟（点校本 校订本） (王钦若, 周勋初) (z-library.sk).pdf",
    "唐摭言 (历代笔记小说大观) (（五代）王定保 撰 阳羡生校点) (z-library.sk).epub",
    "《唐律疏議箋解》 (劉俊文) (z-library.sk).pdf",
    "新唐书 (欧阳修、宋祁) (z-library.sk).epub",
    "通典 (杜佑) (z-library.sk).pdf",
    "阎步克_察举制度变迁史稿.packed",
    "田子爽_唐代制举孝悌类科目考论.pdf",
    "唐令拾遗补 (仁井田陞 著 池田温 补编).pdf",
]
metadata_table: list[dict] = []
for fn in samples:
    md = extract_from_filename(fn)
    layers = ",".join(el.type for el in md.editorial_layers) or "-"
    metadata_table.append({
        "filename": fn,
        "title": md.title,
        "edition": md.edition,
        "primary_author": md.primary_author,
        "secondary_author": md.secondary_author,
        "publication_year": md.publication_year,
        "editorial_layers": layers,
        "confidence": md.confidence,
    })
    print(
        f"  {md.title[:18]:18s} | edition={md.edition[:14]:14s} | "
        f"primary={md.primary_author:8s} | secondary={md.secondary_author:8s} | "
        f"layers={layers:12s} | conf={md.confidence:.2f}"
    )

# Sanity: at least 6 of 8 samples should land an author.
with_author = sum(1 for r in metadata_table if r["primary_author"])
print(f"\n  author coverage on samples: {with_author}/{len(samples)}")
assert with_author >= 6, f"author coverage too low: {with_author}/{len(samples)}"


  册府元龟               | edition=点校本 校订本        | primary=王钦若      | secondary=周勋初      | layers=校訂,校點        | conf=0.40
  唐摭言                | edition=阳羡生校点          | primary=王定保      | secondary=         | layers=校點           | conf=0.40
  唐律疏議箋解             | edition=               | primary=劉俊文      | secondary=         | layers=疏議,箋解        | conf=0.40
  新唐书                | edition=               | primary=欧阳修      | secondary=宋祁       | layers=-            | conf=0.30
  通典                 | edition=               | primary=杜佑       | secondary=         | layers=-            | conf=0.30
  察举制度变迁史稿           | edition=               | primary=阎步克      | secondary=         | layers=-            | conf=0.30
  唐代制举孝悌类科目考论        | edition=               | primary=田子爽      | secondary=         | layers=-            | conf=0.30
  唐令拾遗补              | edition=               | primary=仁井田陞     | secondary=池田温      | layers=校訂           | conf=0.40

  author coverage on samples: 8/8


## 5. Reader probes — text-vs-image classification

Run each reader's `quick_summary` on its fixture so the
"format-aware routing" decision is visible. Three properties matter:

- **page_count** (PDF: real pages; EPUB: spine items; packed-md: .md files).
- **native_pages_in_sample** vs **ocr_pages_in_sample** — the key signal:
  this drives whether the OCR queue (Phase 3) ever fires.
- **mean_chars_per_sampled_page** — sanity for the 200-char threshold.


In [7]:
from apps.backend.readers import pdf_reader, epub_reader, packed_md

EPUB_FIXTURE = REPO_ROOT / "raw/Primary/唐摭言.epub"
PDF_FIXTURE  = REPO_ROOT / (
    "raw/Secondary/specific_科举/田子爽_唐代制举孝悌类科目考论.pdf"
)
PACKED_FIXTURE = REPO_ROOT / (
    "raw/Secondary/comprehensive_epub/阎步克_察举制度变迁史稿.packed"
)

for fix in (EPUB_FIXTURE, PDF_FIXTURE, PACKED_FIXTURE):
    assert fix.exists(), f"fixture missing: {fix}"

probes = {
    "epub": epub_reader.quick_summary(EPUB_FIXTURE, sample_pages=20),
    "pdf":  pdf_reader.quick_summary(PDF_FIXTURE, sample_pages=20),
    "packed_md": packed_md.quick_summary(PACKED_FIXTURE, sample_pages=20),
}
for backend, p in probes.items():
    print(
        f"  {backend:10s} pages={p['page_count']:5d} "
        f"native/ocr={p['native_pages_in_sample']}/{p['ocr_pages_in_sample']} "
        f"mean_chars={p['mean_chars_per_sampled_page']:.0f}  "
        f"tier={p['tier']}"
    )


AssertionError: fixture missing: /Users/mohasani/Ancient/raw/Primary/唐摭言 (历代笔记小说大观) (（五代）王定保 撰 阳羡生校点) (z-library.sk, 1lib.sk, z-lib.sk)(1).epub

## 6. Ingest fixture A — Primary EPUB (`唐摭言`)

First-pass ingest (max 12 pages) of the Tang 笔记小说. Expected:

- tier = `primary`.
- Mix of `native_text` and `ocr` pages (the EPUB embeds scanned-image
  chapter bundles; the reader expands one PageRecord per `<img>`).
- Cover image uploaded to `ancient-pages/` under `<doc_id>/cover.jpg`.
- Editorial layer `校點` detected on the DOCUMENT.


In [ ]:
from apps.backend.pipeline.ingest import ingest_path, upsert_topic, list_ingested_documents

topic = upsert_topic(driver, topic_id="tang-corpus", name="Tang corpus",
                    description="Tang-era primaries + secondaries (v1)")
print("topic:", topic)

report_epub = ingest_path(
    EPUB_FIXTURE,
    driver=driver,
    topic_id="tang-corpus",
    minio_client=minio_client,
    bucket=bucket,
    repo_root=REPO_ROOT,
    max_pages=12,
)
print(f"\n  doc_id={report_epub.document_id}")
print(f"  tier={report_epub.tier} backend={report_epub.backend}")
print(f"  pages total/native/ocr/uploaded = "
      f"{report_epub.pages_total}/{report_epub.pages_native}/"
      f"{report_epub.pages_ocr}/{report_epub.pages_uploaded_to_minio}")
print(f"  duration={report_epub.duration_seconds}s")
print(f"  metadata: {report_epub.metadata['title']!r} edition={report_epub.metadata['edition']!r} "
      f"layers={[el['type'] for el in report_epub.metadata['editorial_layers']]}")
if report_epub.errors:
    print(f"  errors: {report_epub.errors}")

assert report_epub.tier == "primary"
assert report_epub.pages_total > 0
assert report_epub.pages_uploaded_to_minio == report_epub.pages_ocr, (
    "every ocr page must be uploaded to minio for native+image-bundle EPUBs"
)
layers = {el["type"] for el in report_epub.metadata["editorial_layers"]}
assert "校點" in layers, f"expected 校點 layer for 唐摭言 (阳羡生校点); got {layers}"


topic: {'user_id': 'system', 'topic_id': 'tang-corpus'}



  doc_id=唐摭言_(历代笔记小说大观)_(（五代）王定保_撰_阳羡生校点)_(z-library.sk,_1lib.sk,_z-l__59ec485d08
  tier=primary backend=epub
  pages total/native/ocr/uploaded = 12/3/9/9
  duration=0.482s
  metadata: '唐摭言' edition='阳羡生校点' layers=['校點']


## 7. Ingest fixture B — Secondary PDF (modern academic paper)

All-native PDF: PyMuPDF should classify every page as `native_text` and
no MinIO uploads should fire. The DOCUMENT is tagged
`tier=secondary` and the section/cite-style text is preserved verbatim
on `PAGE.text` for downstream chunking.


In [ ]:
report_pdf = ingest_path(
    PDF_FIXTURE,
    driver=driver,
    topic_id="tang-corpus",
    minio_client=minio_client,
    bucket=bucket,
    repo_root=REPO_ROOT,
    max_pages=4,
    reader_kwargs={"target_dpi": 300},
)
print(f"  doc_id={report_pdf.document_id}")
print(f"  tier={report_pdf.tier} backend={report_pdf.backend}")
print(f"  pages total/native/ocr = "
      f"{report_pdf.pages_total}/{report_pdf.pages_native}/{report_pdf.pages_ocr}")
print(f"  duration={report_pdf.duration_seconds}s")
print(f"  metadata: {report_pdf.metadata['title']!r} primary={report_pdf.metadata['primary_author']!r}")
if report_pdf.errors:
    print(f"  errors: {report_pdf.errors}")

assert report_pdf.tier == "secondary"
assert report_pdf.pages_native == report_pdf.pages_total, (
    "modern academic PDF should be 100% native text"
)
assert report_pdf.pages_ocr == 0


  doc_id=田子爽_唐代制举孝悌类科目考论__f48bda28ae
  tier=secondary backend=pdf
  pages total/native/ocr = 4/4/0
  duration=0.092s
  metadata: '唐代制举孝悌类科目考论' primary='田子爽'


## 8. Ingest fixture C — Secondary packed-markdown tree

`阎步克_察举制度变迁史稿.packed/` is the Obsidian-Epub-Importer pre-chunk:
one Markdown file per section, in directories that mirror the book's
part/chapter hierarchy. The reader preserves that hierarchy on
`PAGE.metadata.section_path`; the Phase-1 structure extractor
(`apps.backend.pipeline.structure.plan_structure`) reads that list to
wire `(:CHAPTER)-[:INCLUDE]->(:SECTION)-[:INCLUDE]->(:PAGE)` directly —
top-level directory under `.packed/` becomes a CHAPTER, second-level
becomes a SECTION (plan §6 Stage 1).


In [ ]:
report_packed = ingest_path(
    PACKED_FIXTURE,
    driver=driver,
    topic_id="tang-corpus",
    minio_client=minio_client,
    bucket=bucket,
    repo_root=REPO_ROOT,
    max_pages=15,
)
print(f"  doc_id={report_packed.document_id}")
print(f"  tier={report_packed.tier} backend={report_packed.backend}")
print(f"  pages total/native/ocr = "
      f"{report_packed.pages_total}/{report_packed.pages_native}/{report_packed.pages_ocr}")
print(f"  duration={report_packed.duration_seconds}s")
print(f"  metadata: {report_packed.metadata['title']!r} primary={report_packed.metadata['primary_author']!r}")
if report_packed.errors:
    print(f"  errors: {report_packed.errors}")

assert report_packed.tier == "secondary"
assert report_packed.backend == "packed_md"
assert report_packed.pages_native == report_packed.pages_total


  doc_id=阎步克_察举制度变迁史稿__54ea7f43bc
  tier=secondary backend=packed_md
  pages total/native/ocr = 15/15/0
  duration=0.295s
  metadata: '察举制度变迁史稿' primary='阎步克'


## 9. Optional — LLM enrichment for ambiguous filenames

For filenames where the regex layer is uncertain (no parens, modern
academic shape), call `llm_enrich(...)` to ask `deepseek-chat` for
structured JSON. We pick `阎步克_察举制度变迁史稿.packed` (regex confidence
0.30, no editorial layers detected) as the demo case. The notebook
tolerates Silra failures: it prints a warning and falls back to the
base regex extraction.


In [ ]:
from apps.backend.pipeline.metadata import extract_from_filename, llm_enrich

target_filename = "阎步克_察举制度变迁史稿.packed"
base = extract_from_filename(target_filename)

snippet = ""
with driver.session() as session:
    row = session.run(
        '''
        MATCH (d:DOCUMENT {id: $doc_id})-[:CONSIST_OF]->(:CHAPTER)-[:INCLUDE]->(:SECTION)-[:INCLUDE]->(p:PAGE)
        WHERE p.text IS NOT NULL
        RETURN p.text AS text
        ORDER BY p.docPageIndex
        LIMIT 1
        ''',
        doc_id=report_packed.document_id,
    ).single()
    if row:
        snippet = (row["text"] or "")[:600]

try:
    enriched = llm_enrich(target_filename, snippet=snippet, base=base, max_retries=1)
    print(f"  base    title={base.title!r}     primary={base.primary_author!r}     conf={base.confidence:.2f}")
    print(f"  enriched title={enriched.title!r} primary={enriched.primary_author!r} conf={enriched.confidence:.2f}")
    print(f"  publisher={enriched.publisher!r} year={enriched.publication_year}")
    print(f"  via={enriched.extracted_via}")
    llm_demo = {"ok": True, "filename": target_filename, "base": base.to_dict(), "enriched": enriched.to_dict()}
except Exception as exc:
    import traceback
    print(f"  llm_enrich raised {type(exc).__name__}: {exc}")
    traceback.print_exc(limit=2)
    llm_demo = {"ok": False, "filename": target_filename, "error": str(exc), "base": base.to_dict()}


  base    title='察举制度变迁史稿'     primary='阎步克'     conf=0.30
  enriched title='察举制度变迁史稿' primary='阎步克' conf=0.30
  publisher='' year=None
  via=['filename_regex', 'deepseek-chat']


## 10. Verify Neo4j state

Roll up the topic-scoped document inventory (one row per DOCUMENT) and
cross-check page totals against the per-document reports.


In [ ]:
docs = list_ingested_documents(driver, topic_id="tang-corpus")
print(f"documents in topic 'tang-corpus': {len(docs)}")
for d in docs:
    print(f"  - id={d['id']}")
    print(f"    title={d['title']!r} tier={d['tier']} backend={d['backend']} pages={d['page_count']}")

with driver.session() as session:
    page_total = session.run("MATCH (:PAGE) RETURN count(*) AS n").single()["n"]
    next_links = session.run("MATCH (:PAGE)-[:NEXT]->(:PAGE) RETURN count(*) AS n").single()["n"]
print(f"\nNeo4j totals: pages={page_total}, NEXT links={next_links}")

expected_pages = report_epub.pages_total + report_pdf.pages_total + report_packed.pages_total
assert page_total >= expected_pages, (
    f"expected at least {expected_pages} PAGE nodes after ingest, got {page_total}"
)


Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: publicationYear)", position=<SummaryInputPosition line=8, column=14, offset=324>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'column': 14, 'offset': 324, 'line': 8}}> for query: '\n    MATCH (d:Document)\n    OPTIONAL MATCH (d)-[:HAS_PAGE]->(p:Page)\n    WITH d, count(p) AS page_count\n    WHERE EXISTS { MATCH (:Topic {id: $topic_id})-[:CONTAIN]->(d) } \n    RETURN d.id AS id, d.title A

documents in topic 'tang-corpus': 3
  - id=田子爽_唐代制举孝悌类科目考论__f48bda28ae
    title='唐代制举孝悌类科目考论' tier=secondary backend=pdf pages=4
  - id=唐摭言_(历代笔记小说大观)_(（五代）王定保_撰_阳羡生校点)_(z-library.sk,_1lib.sk,_z-l__59ec485d08
    title='唐摭言' tier=primary backend=epub pages=12
  - id=阎步克_察举制度变迁史稿__54ea7f43bc
    title='察举制度变迁史稿' tier=secondary backend=packed_md pages=15

Neo4j totals: pages=31, NEXT links=28


## 11. Verify MinIO state

The EPUB fixture is the only one with image-bundle pages; we expect
its document_id prefix in the bucket and roughly
`report_epub.pages_uploaded_to_minio` keys.


In [ ]:
objects = list(minio_client.list_objects(bucket, prefix=report_epub.document_id, recursive=True))
print(f"minio objects under {report_epub.document_id!r}: {len(objects)}")
for obj in objects[:5]:
    print(f"  {obj.object_name}  ({obj.size} bytes)")
if len(objects) > 5:
    print(f"  ... +{len(objects)-5} more")

assert len(objects) >= report_epub.pages_uploaded_to_minio, (
    f"expected ≥{report_epub.pages_uploaded_to_minio} objects, found {len(objects)}"
)


minio objects under '唐摭言_(历代笔记小说大观)_(（五代）王定保_撰_阳羡生校点)_(z-library.sk,_1lib.sk,_z-l__59ec485d08': 9
  唐摭言_(历代笔记小说大观)_(（五代）王定保_撰_阳羡生校点)_(z-library.sk,_1lib.sk,_z-l__59ec485d08/cover.jpg  (110786 bytes)
  唐摭言_(历代笔记小说大观)_(（五代）王定保_撰_阳羡生校点)_(z-library.sk,_1lib.sk,_z-l__59ec485d08/page_00004.jpg  (2643 bytes)
  唐摭言_(历代笔记小说大观)_(（五代）王定保_撰_阳羡生校点)_(z-library.sk,_1lib.sk,_z-l__59ec485d08/page_00005.jpg  (2768 bytes)
  唐摭言_(历代笔记小说大观)_(（五代）王定保_撰_阳羡生校点)_(z-library.sk,_1lib.sk,_z-l__59ec485d08/page_00006.jpg  (2868 bytes)
  唐摭言_(历代笔记小说大观)_(（五代）王定保_撰_阳羡生校点)_(z-library.sk,_1lib.sk,_z-l__59ec485d08/page_00007.jpg  (3075 bytes)
  ... +4 more


## 12. Save artifact + handoff

Persist the per-document reports to `_artifacts/01_ingestion/ingestion.json`
so downstream notebooks (02 preprocessing, 03 OCR, 1b language detection,
1c bulk loader) can consume the inventory without re-walking `raw/`.


In [ ]:
artifact = {
    "phase": "01_ingestion",
    "topic_id": "tang-corpus",
    "neo4j_totals": {
        "pages": page_total,
        "next_links": next_links,
        "documents": len(docs),
    },
    "documents": [
        report_epub.to_dict(),
        report_pdf.to_dict(),
        report_packed.to_dict(),
    ],
    "metadata_samples": metadata_table,
    "reader_probes": probes,
    "llm_enrich_demo": llm_demo,
    "fixtures": {
        "epub": str(EPUB_FIXTURE.relative_to(REPO_ROOT)),
        "pdf":  str(PDF_FIXTURE.relative_to(REPO_ROOT)),
        "packed_md": str(PACKED_FIXTURE.relative_to(REPO_ROOT)),
    },
}
out = ARTIFACT_DIR / "ingestion.json"
out.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f"wrote {out} ({out.stat().st_size} bytes)")
print("next: 01b_language_detection_native.ipynb")


wrote /Users/mohasani/Ancient/notebooks/_artifacts/01_ingestion/ingestion.json (9663 bytes)
next: 01b_language_detection_native.ipynb
